In [11]:
import os
import sys
import time
import yaml
import pandas as pd
import numpy as np
import json

with open('../../config.local.yaml', 'r') as f:
    local_config = yaml.safe_load(f)

LOCAL_PATH = local_config['LOCAL_PATH']
DATA_PATH = local_config['DATA_PATH']

sys.path.append(os.path.join(LOCAL_PATH, "src/python"))

import llm_tools as llt


In [12]:
MODEL = "claude-sonnet-4-6"
LLM_PROVIDER = "claude" # openai | claude
MAX_OUTPUT_TOKENS = 4096
input_cost = 1.50 # batch cost per 1M tokens
output_cost = 7.50 # batch cost per 1M tokens
output_tokens_per_request = 500
REQUESTS_PER_BATCH = 4000

In [13]:
MIN_IDX = 0
MAX_IDX = 250
REPLACE = False
REPLACE_OUTPUT_FILE = True
MODE = 'WRITE'  # ESTIMATE | BATCH | WRITE

In [14]:
meetings_df = pd.read_csv(os.path.join(DATA_PATH, "intermediate_data/cpc/meetings-manifest.csv"))
DATES = sorted(list(meetings_df['date']))

In [15]:
PROMPT = r"""
The following text contains an agenda item from an LA City Planning Commission meeting. 

Read the agenda item carefully and extract the relevant information as specified below.

Return a JSON in the following format:

{
    "item_no": "<agenda item number, not including the period, e.g. '1', '2', '3a', '3b', etc.>",
    "title": "<title of agenda item>",
    "address": "<address or location of agenda item if applicable, empty string if not>",
    "council_district": "<council district number(s), CITYWIDE, or empty string>",
    "council_member": "<name(s) of council district member(s), CITYWIDE, or empty string>",
    "last_day_to_act": "<YYYY-MM-DD, or empty string>",
    "referenced_laws": ["<list any referenced laws, codes, ordinances, or programs>"] | empty list if none,
    "is_appeal": "<whether item is an appeal of a previous decision: yes|no|n/a>",
    "summary_of_appeal": "<short summary of the appeal if applicable, empty string if not>",
    "summary": "<short summary of agenda item>"
}

Guidelines:
- Return valid JSON only, no other text or markdown formatting.
- Agenda titles are often Planning Department case numbers like "CPC-2023-1234-DB-SPR", or generic matters of business like "Director's Report and Commission Business" or "General Public Comment".
- When the agenda item is a planning case, focus the summary on the key aspects of the case, such as the project type, location, and requested actions.
- If there is an appeal, summarize the grounds of the appeal and note which requested actions are being appealed.

AGENDA:

"""

In [16]:
# write prompt to figures
with open(os.path.join(LOCAL_PATH, 'figures', 'agenda_split_prompt.tex'), 'w') as f:
    out = PROMPT + "<<agenda text>>"
    out = out.strip()
    out = out.replace('\n', '\\\\ \n')
    f.write(out)

In [17]:
input_tokens = 0
output_tokens = 0
n_requests = 0
n_direct = 0
batch_num = 0

rel_path = os.path.join(DATA_PATH, "response_store")
batch_db_path = os.path.join(rel_path, "batch_jobs.db")
response_db_path = os.path.join(rel_path, "responses.db")

batch_db_conn = llt.get_batch_jobs_db_conn(batch_db_path)
response_db_conn = llt.get_response_store_db_conn(response_db_path)

t0 = time.time()
prompts_to_submit = []
for idx, row in meetings_df.iterrows():
    if idx < MIN_IDX or idx > MAX_IDX:
        continue

    date = row['date']
    year = date[0:4]

    print(f"{idx} {date} ...")

    # Input data
    input_filepath = os.path.join(DATA_PATH, "intermediate_data/cpc", year, date, "agenda-cleaned.json")
    override_filepath = os.path.join(LOCAL_PATH, "intermediate_data/cpc", year, date, "agenda-cleaned-override.json")
    if (not os.path.exists(input_filepath)) and (not os.path.exists(override_filepath)):
        print("No agenda text file found, skipping.")
        continue
    if os.path.exists(override_filepath):
        with open(override_filepath, 'r', encoding='utf-8') as f:
            j = json.load(f)
    else:
        with open(input_filepath, 'r', encoding='utf-8') as f:
            j = json.load(f)

    # Output paths
    output_dir = os.path.join(DATA_PATH, "intermediate_data/cpc", year, date)
    os.makedirs(output_dir, exist_ok=True)
    output_filepath = os.path.join(output_dir, "agenda-cleaned-summarized.json")
    if (not REPLACE_OUTPUT_FILE) and os.path.exists(output_filepath):
        print("Output file exists, skipping.")
        continue

    for k, v in enumerate(j):
        j[k]["ai_summary"] = ""
        item_no = j[k]["item_no"]
        sub_item_no = j[k]["sub_item_no"]
        text = j[k]["text"]
        is_consent_calendar = j[k]["is_consent_calendar"]
        if is_consent_calendar and sub_item_no == "":   # skip consent calendar headers
            continue

        prompt = PROMPT + text

        # check if prompt is in cache
        if not REPLACE:
            prompt_hash = llt.get_hash(prompt)
            cached_response = llt.check_response_cache(prompt_hash, response_db_conn)
            if cached_response:
                if MODE=='WRITE':
                    response = cached_response[0]
                    logprob = cached_response[1]
                    perplexity = cached_response[2]
                    parsed = json.loads(response.strip().removeprefix("```json").removesuffix("```").strip())
                    j[k]["ai_summary"] = parsed
                continue

        #my_tokens = llt.count_tokens(prompt, MODEL)
        input_tokens += len(prompt)/3.5
        output_tokens += output_tokens_per_request
        n_requests += 1

        # add prompt to batch and check if batch is ready to submit
        if MODE=='BATCH':
            prompts_to_submit.append(prompt)
            if len(prompts_to_submit) >= REQUESTS_PER_BATCH:
                input_filename = f"summarize_agendas_batch_{batch_num}.jsonl"
                llt.create_chat_batch_job(prompts_to_submit, rel_path, input_filename, model=MODEL, batch_db_conn=batch_db_conn, response_db_conn=response_db_conn, overwrite=REPLACE, provider=LLM_PROVIDER, max_output_tokens=MAX_OUTPUT_TOKENS)
                prompts_to_submit = []
                batch_num += 1
                print(f"    batch submitted")
        elif MODE=='WRITE':
            my_response = llt.get_response(prompt, MODEL, response_db_conn, overwrite=REPLACE, provider=LLM_PROVIDER, max_output_tokens=MAX_OUTPUT_TOKENS)
            response = my_response['response']
            logprob = my_response['total_logprob']
            perplexity  = my_response['perplexity']
            n_direct += 1
            parsed = json.loads(response.strip().removeprefix("```json").removesuffix("```").strip())
            j[k]["ai_summary"] = parsed
    
    if MODE=='WRITE':
        # write output file
        with open(output_filepath, 'w', encoding='utf-8') as f:
            json.dump(j, f, indent=2, ensure_ascii=False)
        print(f"    wrote output file")

if (MODE=='BATCH') and (len(prompts_to_submit) > 0):
    input_filename = f"summarize_agendas_batch_{batch_num}.jsonl"
    llt.create_chat_batch_job(prompts_to_submit, rel_path, input_filename, model=MODEL, batch_db_conn=batch_db_conn, response_db_conn=response_db_conn, overwrite=REPLACE, provider=LLM_PROVIDER, max_output_tokens=MAX_OUTPUT_TOKENS)
    prompts_to_submit = []
    print(f"    batch submitted")

t1 = time.time()
print(f"    elapsed time: {(t1-t0)/60:.2f} minutes, {n_direct:,} direct requests")

batch_db_conn.close()
response_db_conn.close()

print(f"{n_direct:,} direct requests made")

0 2018-05-10 ...
    wrote output file
1 2018-05-23 ...
    wrote output file
2 2018-06-14 ...
    wrote output file
3 2018-07-12 ...
    wrote output file
4 2018-07-26 ...
    wrote output file
5 2018-08-09 ...
    wrote output file
6 2018-08-23 ...
    wrote output file
7 2018-09-13 ...
    wrote output file
8 2018-09-27 ...
    wrote output file
9 2018-10-11 ...
    wrote output file
10 2018-10-25 ...
    wrote output file
11 2018-11-08 ...
    wrote output file
12 2018-11-29 ...
    wrote output file
13 2018-12-13 ...
    wrote output file
14 2018-12-20 ...
    wrote output file
15 2019-01-10 ...
    wrote output file
16 2019-01-24 ...
    wrote output file
17 2019-02-14 ...
    wrote output file
18 2019-02-28 ...
    wrote output file
19 2019-03-14 ...
    wrote output file
20 2019-03-28 ...
    wrote output file
21 2019-04-11 ...
    wrote output file
22 2019-05-09 ...
    wrote output file
23 2019-05-23 ...
    wrote output file
24 2019-06-13 ...
    wrote output file
25 2019-06

In [18]:
print(date)

2024-12-19


In [19]:
# Cost Estimate

total_input_cost = input_tokens / 1e6 * input_cost
total_output_cost = output_tokens / 1e6 * output_cost

print(f"Estimated number of requests: {n_requests:,.0f}")
print(f"Estimated total input tokens: {input_tokens:,.0f}")
print(f"Estimated total input cost: ${total_input_cost:.2f}")
print(f"Estimated total output tokens: {n_requests * output_tokens_per_request:,.0f}")
print(f"Estimated total output cost: ${total_output_cost:.2f}")
print(f"Estimated total cost: ${total_input_cost + total_output_cost:.2f}")

Estimated number of requests: 0
Estimated total input tokens: 0
Estimated total input cost: $0.00
Estimated total output tokens: 0
Estimated total output cost: $0.00
Estimated total cost: $0.00


In [20]:
for item in j:
    print(json.dumps(item["ai_summary"], indent=2))

{
  "item_no": "1",
  "title": "DIRECTOR'S REPORT AND COMMISSION BUSINESS",
  "address": "",
  "council_district": "",
  "council_member": "",
  "last_day_to_act": "",
  "referenced_laws": [],
  "is_appeal": "n/a",
  "summary_of_appeal": "",
  "summary": "Routine commission business including legal actions and issues update, items of interest, advance calendar, commission requests, and approval of meeting minutes from January 25, 2024."
}
{
  "item_no": "2",
  "title": "NEIGHBORHOOD COUNCIL POSITION STATEMENTS ON AGENDA ITEMS",
  "address": "",
  "council_district": "",
  "council_member": "",
  "last_day_to_act": "",
  "referenced_laws": [],
  "is_appeal": "n/a",
  "summary_of_appeal": "",
  "summary": "Opportunity for Neighborhood Council representatives to present resolutions or community impact statements related to any item on the current agenda. Representatives must provide copies to the Commission via email, and presentations may be taken at the Chair's discretion when the relat